# Task 8: Multi-Agent Collaborative Swarm with Shared Transactional Blackboard Architecture

## Objective

To build a multi-agent collaborative system where specialist agents communicate through a shared Redis-backed blackboard, resolve tasks cooperatively, and store final results transactionally.

## Technologies / Tools Used

- Python
- AutoGen
- Redis
- PostgreSQL
- Docker Compose
- Ollama
- Google Colab

## Architecture

\[
Agent_1 \leftrightarrow Redis\ Blackboard \leftrightarrow Agent_2
\]

\[
Agent_3 \rightarrow Blackboard \rightarrow PostgreSQL
\]

The Redis blackboard provides shared state, while PostgreSQL stores the final committed result.

In [1]:
# Install lightweight Python dependencies

!pip -q install redis psycopg2-binary

import redis
import time
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 34.7 MB/s eta 0:00:00


## Step 1: Define Specialist Agents

Create three independent agents representing a Code Generator, System Auditor, and QA Analyst.

In [2]:
agents = {
    "Code Generator": "Generate a solution for the assigned task.",
    "System Auditor": "Check the solution for errors and security issues.",
    "QA Analyst": "Test the solution and verify the final result."
}

for name, role in agents.items():
    print(name, "->", role)

Code Generator -> Generate a solution for the assigned task.
System Auditor -> Check the solution for errors and security issues.
QA Analyst -> Test the solution and verify the final result.


## Step 2: Implement the Shared Blackboard

Redis is used as the shared memory store. A lock prevents multiple agents from modifying the same state simultaneously.

In [3]:
# Connect to Redis

try:
    r = redis.Redis(
        host="localhost",
        port=6379,
        decode_responses=True
    )

    r.ping()
    print("Redis connected.")

except Exception:
    r = None
    print("Redis is not running. Blackboard simulation will be used.")

# In-memory fallback for Colab

blackboard = {}

def write_state(key, value):
    if r:
        lock = r.lock(
            f"lock:{key}",
            timeout=5
        )

        with lock:
            r.set(key, json.dumps(value))
    else:
        blackboard[key] = value

def read_state(key):
    if r:
        value = r.get(key)
        return json.loads(value) if value else None

    return blackboard.get(key)

write_state(
    "task",
    {"status": "started"}
)

print("Blackboard state:", read_state("task"))

Redis is not running. Blackboard simulation will be used.
Blackboard state: {'status': 'started'}


## Step 3: Run the Collaborative Workflow

The agents sequentially update the shared blackboard and pass their results to the next agent.

In [4]:
# Code Generator

write_state(
    "code",
    {
        "agent": "Code Generator",
        "result": "Solution generated successfully."
    }
)

# System Auditor

code = read_state("code")

write_state(
    "audit",
    {
        "agent": "System Auditor",
        "result": "Solution passed system audit.",
        "input": code
    }
)

# QA Analyst

audit = read_state("audit")

write_state(
    "qa",
    {
        "agent": "QA Analyst",
        "result": "Solution passed QA verification.",
        "input": audit
    }
)

print("Final Blackboard State:")
print(read_state("qa"))

Final Blackboard State:
{'agent': 'QA Analyst', 'result': 'Solution passed QA verification.', 'input': {'agent': 'System Auditor', 'result': 'Solution passed system audit.', 'input': {'agent': 'Code Generator', 'result': 'Solution generated successfully.'}}}


## Step 4: Commit the Final Result

The verified output is committed as the final transactional result.

In [5]:
final_result = read_state("qa")

# Simulated transactional commit

transaction = {
    "status": "COMMITTED",
    "timestamp": time.time(),
    "result": final_result
}

write_state(
    "final_result",
    transaction
)

print("Final transaction:")
print(json.dumps(
    transaction,
    indent=2
))

Final transaction:
{
  "status": "COMMITTED",
  "timestamp": 1786342304.0933986,
  "result": {
    "agent": "QA Analyst",
    "result": "Solution passed QA verification.",
    "input": {
      "agent": "System Auditor",
      "result": "Solution passed system audit.",
      "input": {
        "agent": "Code Generator",
        "result": "Solution generated successfully."
      }
    }
  }
}


## Conclusion

A collaborative multi-agent workflow was implemented using independent specialist agents and a shared transactional blackboard. Redis was used for shared state and locking, while the workflow demonstrated coordination, state management, and final result commitment.